# Attn-1DCNN-EE: Nuclear Power Plant Fault Detection
## Attention-based 1D-CNN with Elliptic Envelope for Open-Set Anomaly Detection

---

This notebook runs the full **Attn-1DCNN-EE** pipeline on the NPPAD dataset:

| Phase | Description |
|-------|-------------|
| **Setup** | Clone repo, install dependencies, upload data |
| **Data Pipeline** | Load CSVs, clean, scale, sliding-window segmentation |
| **Phase 1** | Supervised training (CNN + Attention + CrossEntropy) |
| **Phase 2** | Fit per-class Elliptic Envelopes on learned features |
| **Evaluation** | Closed-set accuracy + open-set unknown detection |
| **XAI** | Attention heatmaps, SHAP attribution, faithfulness, LLM report |

### 0. GPU & RAM Check

In [ ]:
# ── GPU & RAM check ──────────────────────────────────────────────
# Run this first to confirm Colab has assigned a GPU.
# If you see 'No GPU' go to Runtime → Change runtime type → T4 GPU.
import subprocess, textwrap

print('=' * 55)
print(' ENVIRONMENT CHECK')
print('=' * 55)

# GPU
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
         '--format=csv,noheader'],
        capture_output=True, text=True, check=True
    )
    print('GPU :', result.stdout.strip())
except Exception:
    print('GPU : ❌  No GPU detected — change runtime to T4/A100!')

# RAM
try:
    with open('/proc/meminfo') as f:
        lines = f.readlines()
    mem = {l.split(':')[0]: l.split(':')[1].strip() for l in lines}
    total_gb = int(mem['MemTotal'].split()[0]) / 1024 / 1024
    avail_gb = int(mem['MemAvailable'].split()[0]) / 1024 / 1024
    print(f'RAM : {total_gb:.1f} GB total, {avail_gb:.1f} GB available')
except Exception:
    print('RAM : (could not read /proc/meminfo)')

# Disk
try:
    result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
    lines = result.stdout.strip().splitlines()
    disk_info = lines[1].split() if len(lines) > 1 else []
    if disk_info:
        print(f'Disk: {disk_info[3]} available of {disk_info[1]} total')
except Exception:
    pass

print('=' * 55)


## 1. Setup

### 1a. (Optional) Mount Google Drive

Mount your Drive to **persist checkpoints** across Colab sessions.
Skip this cell if you don't need persistence — Colab's `/content` is
ephemeral and will be wiped when the runtime disconnects.

After mounting, set `SAVE_DIR` in the *Save Artifacts* cell to a
Drive path such as `'/content/drive/MyDrive/Attn-1DCNN-EE/saved_models'`.

In [ ]:
# Uncomment the lines below to mount your Google Drive.
# Once mounted your files will persist at /content/drive/MyDrive/

# from google.colab import drive
# drive.mount('/content/drive')

# Optional: point your save directory to Drive
# SAVE_DIR = '/content/drive/MyDrive/Attn-1DCNN-EE/saved_models'
# import os; os.makedirs(SAVE_DIR, exist_ok=True)
# print(f'Drive mounted — saving artifacts to: {SAVE_DIR}')

print('Drive mount skipped (using /content — ephemeral storage).')


In [ ]:
# Clone the repository and install dependencies
!git clone https://github.com/Sisoodiya/Attn-1DCNN-EE.git
%cd /content/Attn-1DCNN-EE
!pip install -q -r requirements.txt
!pip install -q -U "lightning>=2.4.0" "pytorch-lightning>=2.4.0"

## 2. Upload & Extract Data

Upload a `.zip` file containing your NPPAD data.

**Expected zip structure** (one of these):
```
your_data.zip
  ├── Operation_csv_data/
  │   ├── Normal/
  │   ├── LOCA/
  │   ├── FLB/
  │   └── ... (18 folders)
  └── Dose_csv_data/       (optional)
```

In [ ]:
import zipfile, os, glob
from google.colab import files

print("Select your NPPAD data .zip file:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Extract into data/
os.makedirs("data", exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('data/')

# Auto-detect: if there's a nested folder, adjust path
op_candidates = glob.glob('data/**/Operation_csv_data', recursive=True)
if op_candidates:
    DATA_DIR = op_candidates[0]
    print(f"Found Operation data at: {DATA_DIR}")
else:
    DATA_DIR = 'data/Operation_csv_data'
    print(f"Using default path: {DATA_DIR}")

# Verify
classes_found = sorted(os.listdir(DATA_DIR))
print(f"Accident classes found ({len(classes_found)}): {classes_found}")

## 3. Imports & Configuration

In [ ]:
import logging
import numpy as np
import torch
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings(
    "ignore",
    message=r"`isinstance\(treespec, LeafSpec\)` is deprecated.*",
    category=DeprecationWarning,
)

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import (
        EarlyStopping, ModelCheckpoint, LearningRateMonitor,
    )
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import (
        EarlyStopping, ModelCheckpoint, LearningRateMonitor,
    )

from data_pipeline.dataset_builder import NPPADDataModule
from models import Attn1DCNN_EE

logging.basicConfig(
    level=logging.INFO,
    format='%(name)s \u2014 %(levelname)s \u2014 %(message)s',
)

# Reproducibility
pl.seed_everything(42, workers=True)

print(f"PyTorch  : {torch.__version__}")
print(f"Lightning: {pl.__version__}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ======================== HYPERPARAMETERS ========================
# Adjust these as needed before running the training cells below.

CONFIG = {
    # --- Data pipeline ---
    "data_dir":       DATA_DIR,
    "window_size":    50,
    "stride":         1,
    "batch_size":     64,
    "val_split":      0.15,
    "test_split":     0.15,
    "num_workers":    0,          # safer default on Colab; raise to 2 if stable
    "scaler_type":    "zscore",

    # --- Model ---
    "backbone_channels":     [64, 128, 256],
    "backbone_kernel_sizes": 3,

    # --- Training (Phase 1) ---
    "max_epochs":     50,
    "lr":             1e-3,
    "weight_decay":   1e-4,
    "scheduler":      "cosine",
    "patience":       10,      # early stopping patience

    # --- Elliptic Envelope (Phase 2) ---
    "ee_contamination": 0.01,
    # --------------------------------------------------------
    # RAM tips:
    #   stride=5  → ~5x fewer windows vs stride=1 (recommended)
    #   stride=10 → ~10x fewer windows (fastest run)
    # --------------------------------------------------------
}

print("Configuration ready.")

## 4. Data Pipeline

The `NPPADDataModule` orchestrates 5 stages:
1. **Load** — read all CSVs, build label map, drop TIME column
2. **Clean** — NaN interpolation, Z-score outlier removal
3. **Scale** — per-feature Z-score standardisation
4. **Window** — sliding-window segmentation → `(N, I, F)` arrays
5. **Split** — stratified train / val / test (70 / 15 / 15)

In [ ]:
dm = NPPADDataModule(
    data_dir       = CONFIG["data_dir"],
    window_size    = CONFIG["window_size"],
    stride         = CONFIG["stride"],
    batch_size     = CONFIG["batch_size"],
    val_split      = CONFIG["val_split"],
    test_split     = CONFIG["test_split"],
    num_workers    = CONFIG["num_workers"],
    scaler_type    = CONFIG["scaler_type"],
)

# Run the full pipeline now so we can inspect the data and know
# num_classes / num_features before building the model.
dm.setup()

print(f"Classes     : {dm.num_classes}")
print(f"Features (F): {dm.num_features}")
print(f"Window   (I): {CONFIG['window_size']}")
print(f"Label map   : {dm.label_map}")
print(f"\nSplit sizes:")
print(f"  Train : {len(dm.train_dataset):,}")
print(f"  Val   : {len(dm.val_dataset):,}")
print(f"  Test  : {len(dm.test_dataset):,}")

In [ ]:
# Class distribution in training set
inv_map = {v: k for k, v in dm.label_map.items()}
train_labels = dm.train_dataset.labels.numpy()
unique, counts = np.unique(train_labels, return_counts=True)

fig, ax = plt.subplots(figsize=(12, 5))
names = [inv_map[u] for u in unique]
bars = ax.bar(names, counts, color='#3498db')
ax.set_xlabel('Accident Class')
ax.set_ylabel('Number of Windows')
ax.set_title('Training Set — Class Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Verify a single sample shape
x_sample, y_sample = dm.train_dataset[0]
print(f"\nSingle sample — x: {x_sample.shape}  y: {y_sample}")
print(f"Expected shape: (F={dm.num_features}, I={CONFIG['window_size']})")

## 5. Model Architecture

```
Input (B, F, I)
  ↓ CNN1DBackbone     → (B, 256, I)    feature maps
  ↓ SoftAttention     → (B, 256, I)    amplified features + attention weights
  ↓ GlobalAvgPool1d   → (B, 256)       feature vectors
  ↓ Linear            → (B, classes)   logits  (Phase 1 training)
  ↓ EllipticEnvelope  → class / unknown (Phase 2 inference)
```

In [ ]:
model = Attn1DCNN_EE(
    in_channels          = dm.num_features,
    num_classes          = dm.num_classes,
    backbone_channels    = CONFIG["backbone_channels"],
    backbone_kernel_sizes = CONFIG["backbone_kernel_sizes"],
    lr                   = CONFIG["lr"],
    weight_decay         = CONFIG["weight_decay"],
    scheduler            = CONFIG["scheduler"],
    ee_contamination     = CONFIG["ee_contamination"],
)

# Print architecture
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable:,}")

## 6. Phase 1 — Supervised Training

Train the CNN backbone + Soft Attention using cross-entropy loss.
This teaches the network to produce discriminative, class-separable
feature representations.

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=CONFIG["patience"],
        mode="min",
        verbose=True,
    ),
    ModelCheckpoint(
        monitor="val_loss",
        filename="best-{epoch:02d}-{val_loss:.4f}",
        save_top_k=1,
        mode="min",
    ),
    LearningRateMonitor(logging_interval="epoch"),
]

trainer = pl.Trainer(
    max_epochs=CONFIG["max_epochs"],
    accelerator="auto",
    devices=1,
    callbacks=callbacks,
    log_every_n_steps=10,
    deterministic=True,
)

print("Starting Phase 1 training ...")
trainer.fit(model, datamodule=dm)

In [ ]:
# Load TensorBoard for interactive training curves
%load_ext tensorboard
%tensorboard --logdir lightning_logs

In [ ]:
# Load the best checkpoint
best_ckpt = trainer.checkpoint_callback.best_model_path
print(f"Best checkpoint: {best_ckpt}")

model = Attn1DCNN_EE.load_from_checkpoint(best_ckpt)
model.eval()
print("Best model loaded.")

## 7. Test Evaluation (Closed-Set)

In [ ]:
trainer.test(model, datamodule=dm)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Collect all test predictions from the linear head
model.eval()
model.to('cuda' if torch.cuda.is_available() else 'cpu')

all_preds, all_true = [], []
with torch.no_grad():
    for x, y in dm.test_dataloader():
        x = x.to(model.device)
        logits, _, _ = model(x)
        all_preds.append(logits.argmax(dim=-1).cpu().numpy())
        all_true.append(y.numpy())

all_preds = np.concatenate(all_preds)
all_true  = np.concatenate(all_true)

# Classification report
target_names = [inv_map[i] for i in sorted(inv_map)]
print(classification_report(all_true, all_preds, target_names=target_names))

# Confusion matrix
fig, ax = plt.subplots(figsize=(14, 12))
cm = confusion_matrix(all_true, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=target_names)
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', values_format='d')
ax.set_title('Test Set — Confusion Matrix (Linear Head)')
plt.tight_layout()
plt.show()

## 8. Phase 2 — Elliptic Envelope Fitting

Fit one `EllipticEnvelope` per known class on the pooled feature vectors
extracted from the trained backbone + attention. This enables **open-set**
detection: samples outside all envelopes are flagged as unknown faults.

In [ ]:
ee_head = model.fit_envelope(
    dataloader=dm.train_dataloader(),
    label_map=dm.label_map,
)

print(f"\nEnvelopes fitted for {len(ee_head.envelopes_)} classes.")
print(f"Classes modelled: {sorted(ee_head.envelopes_.keys())}")

## 9. Open-Set Detection Demo

Run the EE head on the test set to see how many samples are classified
as known faults vs flagged as **unknown**.

In [ ]:
# Run open-set predictions on the full test set
test_features, test_labels = model.extract_features(dm.test_dataloader())

ee_preds, ee_unknown = ee_head.predict(test_features)

n_total   = len(ee_preds)
n_unknown = int(ee_unknown.sum())
n_known   = n_total - n_unknown

print(f"Test samples : {n_total}")
print(f"Known faults : {n_known}")
print(f"Unknown flags: {n_unknown}")

# Accuracy on samples classified as known
if n_known > 0:
    known_mask = ~ee_unknown
    known_acc = (ee_preds[known_mask] == test_labels[known_mask]).mean()
    print(f"Accuracy on known-classified samples: {known_acc:.4f}")

# --- Demo with synthetic unknown data ---
print("\n--- Synthetic Unknown Fault Demo ---")
rng = np.random.RandomState(99)
synthetic_unknown = rng.randn(20, test_features.shape[1]).astype(np.float32) * 5
syn_preds, syn_unknown = ee_head.predict(synthetic_unknown)
print(f"Synthetic samples: {len(syn_preds)}")
print(f"Flagged unknown  : {int(syn_unknown.sum())} / {len(syn_preds)}")

## 10. XAI — Attention Heatmaps

Visualise which feature channels and time steps the Soft Attention
mechanism focused on when analysing specific test samples.

In [ ]:
from xai.attention_viz import plot_attention_heatmap, plot_attention_top_channels

# Pick a sample from the test set
sample_idx = 0
x_single, y_single = dm.test_dataset[sample_idx]
x_batch = x_single.unsqueeze(0).to(model.device)  # (1, F, I)

with torch.no_grad():
    logits, attn_weights, pooled = model(x_batch)

attn_np = attn_weights.cpu().numpy()  # (1, I, C)
pred_class = logits.argmax(dim=-1).item()

print(f"True class     : {inv_map[y_single.item()]}")
print(f"Predicted class: {inv_map[pred_class]}")

# Full heatmap (top 30 channels for readability)
plot_attention_heatmap(
    attn_np,
    top_k=30,
    title=f"Attention Heatmap — True: {inv_map[y_single.item()]}, Pred: {inv_map[pred_class]}",
)
plt.show()

# Top-10 channels bar chart
plot_attention_top_channels(attn_np, top_k=10)
plt.show()

## 11. XAI — SHAP Feature Attribution

Use SHAP (KernelExplainer) to compute the marginal contribution of
each pooled feature to the EE head's Mahalanobis-distance scoring.

> **Note:** KernelExplainer is model-agnostic but can be slow.
> We use a small k-means background set and explain a few samples.

In [ ]:
import shap
from xai.shap_explainer import SHAPExplainer

# Build a predict_fn that returns the negative min Mahalanobis distance
# (higher = more likely to be a known class).
def predict_fn(features: np.ndarray) -> np.ndarray:
    distances = ee_head.mahalanobis_distances(features)
    dist_matrix = np.column_stack([distances[c] for c in sorted(distances)])
    return -dist_matrix.min(axis=1)  # negative so higher = closer

# Background: k-means summary of training features
train_features, _ = model.extract_features(dm.train_dataloader())
background = shap.kmeans(train_features, 50)

explainer = SHAPExplainer(predict_fn=predict_fn)

# Explain a small batch of test samples
# ── SHAP safety guard ────────────────────────────────────
# KernelExplainer is model-agnostic but O(n_samples²).
# On Colab free tier (T4, ~12 GB RAM) keep n_explain ≤ 10
# and n_samples ≤ 100 to stay under the 90-minute timeout.
# Increase these on Colab Pro / A100 runtimes.
# ---------------------------------------------------------
n_explain = 10   # ← was 20; lower for free-tier safety
shap_values = explainer.explain(
    X=test_features[:n_explain],
    background=background,
    n_samples=100,    # ← was 200; lower for free-tier safety
)

print(f"SHAP values shape: {np.array(shap_values).shape}")

# Contributors vs offsets (averaged over explained samples)
decomp = explainer.contributors_and_offsets(shap_values, top_k=10)
print("\nTop 10 Contributors (push toward diagnosis):")
for name, val in decomp['contributors']:
    print(f"  {name}: {val:+.4f}")
print("\nTop 10 Offsets (counteract diagnosis):")
for name, val in decomp['offsets']:
    print(f"  {name}: {val:+.4f}")

In [ ]:
# SHAP summary plot
shap.summary_plot(
    shap_values,
    test_features[:n_explain],
    max_display=20,
    show=True,
)

## 12. XAI — Faithfulness Evaluation (MDMC)

Validate that the SHAP-identified features genuinely drive predictions
by masking them and measuring the prediction change.

In [ ]:
from xai.faithfulness import FaithfulnessEvaluator

evaluator = FaithfulnessEvaluator(predict_fn=predict_fn)

results = evaluator.evaluate(
    X=test_features[:n_explain],
    shap_values=np.array(shap_values),
    top_k_ratios=[0.05, 0.10, 0.20],
    baseline="mean",
)

print("Faithfulness Results (higher delta = more faithful):")
print(f"{'Ratio':<10} {'k features':<12} {'MSE':<14} {'MAE':<14}")
print("-" * 50)
for label, metrics in results.items():
    print(f"{label:<10} {metrics['k']:<12} {metrics['mse']:<14.6f} {metrics['mae']:<14.6f}")

## 13. XAI — Diagnostic Report

Build an impact-based diagnostic prompt from the SHAP results and
attention analysis. If you have an LLM API key, plug it in to
generate a full natural-language report.

In [ ]:
# Upload your .env file containing your Google API key
# This file should have: GOOGLE-API = YOUR_KEY_HERE
import os
if not os.path.exists('.env'):
    from google.colab import files
    print('Upload your .env file with your GOOGLE-API key:')
    uploaded = files.upload()
    print('.env uploaded successfully.')
else:
    print('.env already exists.')


In [ ]:
# Install Gemini SDK and dotenv (quiet, idempotent)
!pip install -q google-generativeai python-dotenv


In [ ]:
from xai.report import DiagnosticReporter

# ── Load API key from .env ────────────────────────────────────────
# python-dotenv reads .env safely; the key name 'GOOGLE-API'
# (with a hyphen) is accessed via os.environ — dotenv handles it.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()  # reads .env from the current working directory
    GOOGLE_API_KEY = os.environ.get('GOOGLE-API', '')
    if not GOOGLE_API_KEY:
        raise ValueError('.env found but GOOGLE-API key is empty')
    print(f'Google API key loaded: {GOOGLE_API_KEY[:8]}...{GOOGLE_API_KEY[-4:]}')
except ImportError:
    # python-dotenv not installed yet — install it
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'python-dotenv', 'google-generativeai'],
                   check=True)
    from dotenv import load_dotenv
    load_dotenv()
    GOOGLE_API_KEY = os.environ.get('GOOGLE-API', '')
    print(f'Google API key loaded: {GOOGLE_API_KEY[:8]}...{GOOGLE_API_KEY[-4:]}')

# ── Gemini llm_fn ─────────────────────────────────────────────────
import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)
_gemini_model = genai.GenerativeModel('gemini-2.0-flash')

def llm_fn(system: str, user: str) -> str:
    """Call Gemini Pro with a combined system + user prompt."""
    combined = f"{system}\n\n{user}"
    response = _gemini_model.generate_content(combined)
    return response.text

print('Gemini Pro llm_fn ready.')

# ── (Alternative) OpenAI — uncomment if you prefer GPT instead ────
# from openai import OpenAI
# client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY', ''))
# def llm_fn(system, user):
#     resp = client.chat.completions.create(
#         model='gpt-4o-mini',
#         messages=[{'role':'system','content':system},
#                   {'role':'user','content':user}],
#     )
#     return resp.choices[0].message.content

# ── Reporter ──────────────────────────────────────────────────────
reporter = DiagnosticReporter(llm_fn=llm_fn)

# Pick a test sample to diagnose
diag_idx = 0
pred_label = ee_preds[diag_idx]
pred_name  = inv_map.get(pred_label, 'Unknown Fault') if pred_label != -1 else 'Unknown Fault'

# Identify peak attention timesteps
x_diag, _ = dm.test_dataset[diag_idx]
with torch.no_grad():
    _, attn_w, _ = model(x_diag.unsqueeze(0).to(model.device))
attn_over_time = attn_w[0].cpu().numpy().mean(axis=1)  # (I,)
peak_steps = list(map(int, (-attn_over_time).argsort()[:5]))

report = reporter.generate_report(
    prediction=pred_name,
    contributors=decomp['contributors'][:10],
    offsets=decomp['offsets'][:5],
    attention_peak_timesteps=peak_steps,
    sample_id=diag_idx,
)

print('=' * 60)
print('GEMINI DIAGNOSTIC REPORT')
print('=' * 60)
if report['report']:
    print(report['report'])
else:
    print('--- System Prompt ---')
    print(report['system_prompt'])
    print('\n--- User Prompt ---')
    print(report['user_prompt'])


## 14. Save Artifacts

Save the trained model checkpoint and the fitted EE head for later use.

In [ ]:
import pickle

# Save directory
os.makedirs("saved_models", exist_ok=True)

# 1. Lightning checkpoint (already saved by ModelCheckpoint callback)
print(f"Best Lightning checkpoint: {best_ckpt}")

# 2. Save the fitted EE head (sklearn objects)
ee_path = "saved_models/ee_head.pkl"
with open(ee_path, "wb") as f:
    pickle.dump(ee_head, f)
print(f"EE head saved to: {ee_path}")

# 3. Save label map
import json
label_map_path = "saved_models/label_map.json"
with open(label_map_path, "w") as f:
    json.dump(dm.label_map, f, indent=2)
print(f"Label map saved to: {label_map_path}")

# Download to local machine
from google.colab import files as colab_files
colab_files.download(ee_path)
colab_files.download(label_map_path)
print("\nDone. Download the Lightning checkpoint from the file browser if needed.")